# Mambo Coach AI - Fine-tuning Gemma 4

Pipeline: transformers + peft + trl (sin dependencia de unsloth)

**Requisitos:** GPU NVIDIA con CUDA 8GB+ VRAM

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU CUDA disponible")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Torch: {torch.__version__}")

GPU: NVIDIA GeForce RTX 4060
VRAM: 8.0 GB
Torch: 2.5.1+cu121


In [2]:
import pandas as pd
from datasets import Dataset
from pathlib import Path

csv_path = Path("dataset_entrenador_mambo.csv")
if not csv_path.exists():
    csv_path = Path("../model/dataset_entrenador_mambo.csv")
if not csv_path.exists():
    raise FileNotFoundError("No se encontro dataset_entrenador_mambo.csv")

df = pd.read_csv(csv_path)
print(f"Ejemplos: {len(df)}")

def format_conversation(row):
    return {
        "conversations": [
            {"from": "human", "value": row["prompt"]},
            {"from": "gpt", "value": row["completion"]}
        ]
    }

records = df.apply(format_conversation, axis=1).tolist()
dataset = Dataset.from_list(records)
print(f"Dataset listo: {len(dataset)} ejemplos")

Ejemplos: 100
Dataset listo: 100 ejemplos


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Cargando {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo cargado: {type(model).__name__}")
print(f"VRAM usada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Cargando google/gemma-4-E2B-it...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Modelo cargado: Gemma4ForConditionalGeneration
VRAM usada: 6.28 GB


In [4]:
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=16):
        super().__init__()
        self.original = original_linear
        self.lora_A = nn.Linear(original_linear.in_features, r, bias=False)
        self.lora_B = nn.Linear(r, original_linear.out_features, bias=False)
        self.scaling = alpha / r
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)
        self.original.weight.requires_grad = False
    def forward(self, x):
        out = self.original(x)
        lora_out = self.lora_B(self.lora_A(x.to(self.lora_A.weight.dtype)))
        return out + lora_out.to(out.dtype) * self.scaling

# Apply LoRA only to language_model layers
target_names = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
lm = model.model.language_model
lora_count = 0
lora_modules = []
for name, module in lm.named_modules():
    short = name.split(".")[-1]
    if short in target_names and type(module).__name__ == "Linear4bit":
        lora = LoRALinear(module)
        lora = lora.to(device=model.device)
        # Replace parent's reference
        parts = name.split(".")
        parent = lm
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], lora)
        lora_modules.append((name, lora))
        lora_count += 1

print(f"LoRA injected into {lora_count} modules")

# Freeze ALL params first, then unfreeze only LoRA
for param in model.parameters():
    param.requires_grad = False
for name, lora in lora_modules:
    lora.lora_A.weight.requires_grad = True
    lora.lora_B.weight.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Free VRAM: remove vision/audio towers (not needed for text fine-tuning)
import gc
del model.model.vision_tower
del model.model.audio_tower
del model.model.embed_vision
del model.model.embed_audio
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

LoRA injected into 205 modules
Trainable: 24,158,208 / 3,960,178,208 (0.61%)


VRAM after cleanup: 6.12 GB


In [5]:
def apply_chat_template(example):
    messages = []
    for turn in example["conversations"]:
        role = "user" if turn["from"] == "human" else "assistant"
        messages.append({"role": role, "content": turn["value"]})
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template)
print("Ejemplo:")
print(dataset[0]["text"][:300])
print("...")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Ejemplo:
<bos><|turn>user
Edad: 25 años | Sexo: Hombre | Altura: 1.75m | Peso: 70kg | Frecuencia de ejercicio: 1-2 veces/semana | Nivel de rendimiento: Bajo | Estilo de entrenamiento: Fuerza | Lugar de entrenamiento: Gimnasio | Tipo de dieta: Omnívora | Consumo de agua: 1.5L/día | Horas de sueño: 7h | Calida
...


In [6]:
from torch.utils.data import DataLoader

# Collect only LoRA parameters for the optimizer
lora_params = []
for name, p in model.named_parameters():
    if p.requires_grad:
        lora_params.append(p)

optimizer = torch.optim.AdamW(lora_params, lr=2e-4, weight_decay=0.01)
total_steps = 80
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4, total_steps=total_steps
)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding=False, return_tensors=None)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
def collate_fn(batch):
    input_ids = [torch.tensor(b["input_ids"]) for b in batch]
    attention_mask = [torch.ones_like(ids) for ids in input_ids]
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": input_ids.clone()}

dataloader = DataLoader(tokenized_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

model.train()
total_loss = 0
torch.cuda.empty_cache()
print(f"Entrenando {total_steps} steps con {len(lora_params)} LoRA params...")

for step, batch in enumerate(dataloader):
    if step >= total_steps:
        break
    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["labels"].to(model.device)

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    del outputs
    torch.cuda.empty_cache()

    total_loss += loss.item()
    if (step + 1) % 10 == 0:
        avg = total_loss / (step + 1)
        print(f"  Step {step+1}/{total_steps} | Loss: {loss.item():.4f} | Avg: {avg:.4f}")

avg_loss = total_loss / total_steps
print(f"\nCompletado! Loss promedio: {avg_loss:.4f}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Entrenando 80 steps con 410 LoRA params...


  Step 10/80 | Loss: 3.6144 | Avg: 4.2798


  Step 20/80 | Loss: 1.9898 | Avg: 3.4374


  Step 30/80 | Loss: 1.3902 | Avg: 2.8789


  Step 40/80 | Loss: 2.1317 | Avg: 2.5104


  Step 50/80 | Loss: 1.3082 | Avg: 2.2711


  Step 60/80 | Loss: 1.9294 | Avg: 2.1148


  Step 70/80 | Loss: 0.7328 | Avg: 1.9878


  Step 80/80 | Loss: 1.1107 | Avg: 1.8763

Completado! Loss promedio: 1.8763


In [7]:
import os, json

output_dir = "mambo_coach_lora"
os.makedirs(output_dir, exist_ok=True)

# Save LoRA weights
lora_state = {}
for name, lora in lora_modules:
    lora_state[f"{name}.lora_A"] = lora.lora_A.weight.data.cpu()
    lora_state[f"{name}.lora_B"] = lora.lora_B.weight.data.cpu()
torch.save(lora_state, os.path.join(output_dir, "lora_weights.pt"))

# Save metadata
meta = {
    "base_model": MODEL_NAME,
    "lora_r": 16,
    "lora_alpha": 16,
    "target_modules": list(target_names),
    "num_lora_modules": lora_count,
}
with open(os.path.join(output_dir, "config.json"), "w") as f:
    json.dump(meta, f, indent=2)

tokenizer.save_pretrained(output_dir)
print(f"LoRA guardado en {output_dir}/")
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f)) / (1024*1024)
    print(f"  {f}: {size:.1f} MB")

LoRA guardado en mambo_coach_lora/
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  lora_weights.pt: 92.3 MB
  tokenizer.json: 30.7 MB
  tokenizer_config.json: 0.0 MB


In [8]:
model.eval()

def ask_mambo(question):
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response

print("=" * 60)
print("MAMBO COACH AI - Prueba")
print("=" * 60)

preguntas = [
    "Tengo 30 anios, peso 80kg, quiero ganar musculo. Que rutina me recomiendas?",
    "Cuanta proteina debo comer al dia si peso 70kg?",
    "Tengo dolor en la rodilla, puedo seguir entrenando piernas?",
]

for p in preguntas:
    print(f"\nP: {p}")
    print(f"R: {ask_mambo(p)}")
    print("-" * 60)

MAMBO COACH AI - Prueba

P: Tengo 30 anios, peso 80kg, quiero ganar musculo. Que rutina me recomiendas?


R: Rutina para ganar músculo en 30 años:
1. Frecuencia: 4-5 días/semana.
2. Series/repeticiones: 3-4 series, 8-12 repeticiones.
3. Enfoque: Estructura completa, movimientos compuestos (sentadilla, press, peso muerto), ejercicios de aislamiento.
4. Progresión: Aumentar peso o repeticiones cada semana.
5. Nutrición: 2000-2500 kcal/día, 1.6g/kg de proteína.
6. Descanso: 7-8 horas/noche.
------------------------------------------------------------

P: Cuanta proteina debo comer al dia si peso 70kg?


R: Para 70kg, 1.6g/kg/día = 112g. 1.8g/kg/día = 126g. 2.0g/kg/día = 140g. 1.2g/kg/día = 84g. La mayoría de los atletas necesita 1.6-2.2g/kg. 140g es un buen punto de partida.
------------------------------------------------------------

P: Tengo dolor en la rodilla, puedo seguir entrenando piernas?


R: Rutina de ejercicios: rodilla dolorida: 1. reduce volumen: 3x15 series en lugar de 4x12. 2. cambia ejercicios: sentadilla a prensa, press de piernas a sentadilla libre. 3. mejora técnica: sentadilla con peso ligero, rodillas al pie. 4. fortalece glúteos: hip thrusts, peso en cadera. 5. estira: isquiotibiales, flexores de cadera, cuádriceps. 6. considera: natación, ciclismo, elíptica. 7. consulta fisioterapeuta: diagnóstico y plan específico.
------------------------------------------------------------
